<a href="https://colab.research.google.com/github/charre2021/reasoning_agent_example/blob/main/reasoning_agent_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
from typing import List, Dict, Any
import random
from openai import OpenAI
from google.colab import userdata

class LLM:
    def __init__(self, model : str, api_key : str) -> None:
        self.client = OpenAI(api_key = api_key)
        self.model = model

    def generate(self, prompt : str) -> str:
        response = self.client.responses.create(
            model = self.model,
            input = prompt
        )
        return response.output_text

class LLMAgent:
    def __init__(self, llm, action_space: List[str]) -> None:
        self.llm = llm
        self.action_space = action_space
        self.memory = []
        self.current_goal = None

    def set_goal(self, goal: str) -> None:
        self.current_goal = goal

    def perceive(self, observation: str) -> None:
        self.memory.append(observation)

    def think(self) -> str:
        context = f"Goal: {self.current_goal}\n"
        context += "Recent observations:\n"
        context += "\n".join(self.memory[-5:])
        context += "\nThink about the current situation and the goal. What should be done next?"

        return self.llm.generate(context)

    def decide(self, thought: str) -> str:
        context = f"Thought: {thought}\n"
        context += "Based on this thought, choose the most appropriate action from the following:\n"
        context += ", ".join(self.action_space)
        context += "\nChosen action:"

        return self.llm.generate(context)


    def act(self, action: str) -> str:
        outcomes = [
            f"Action '{action}' was successful.",
            f"Action '{action}' failed.",
            f"Action '{action}' had an unexpected outcome."
        ]
        return random.choice(outcomes)

    def run_step(self):
        thought = self.think()
        action = self.decide(thought)
        outcome = self.act(action)
        self.perceive(outcome)
        return thought, action, outcome



In [ ]:
llm = LLM("o4-mini", userdata.get("openai"))
action_space = ["move", "grab", "drop", "use", "talk"]
agent = LLMAgent(llm, action_space)

agent.set_goal("Find the key and unlock the door.")
agent.perceive("You are in a room with a table and a chair. There's a drawer in the table.")

for _ in range(5):
    thought, action, outcome = agent.run_step()
    print(f"Thought: {thought}")
    print(f"Action: {action}")
    print(f"Outcome: {outcome}")